In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (precision_score, accuracy_score, recall_score, f1_score, roc_auc_score,
                             matthews_corrcoef, cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve)

# Load CSV
df = pd.read_csv('01_AO_db.csv')  # Replace with your actual CSV file path
df.drop('Unnamed: 0',axis=1, inplace=True)

cls = input('FRS or Chelator? ')

def undersampling(df_augmented, cls):
    # Drop sequences with length smaller than 6 and create a new dataframe
    data = df_augmented.copy()
    #data = data[data['sequence'].str.len() <= 32].copy()
    #data_filtered = data[data['sequence'].str.len() > 7].copy()
    #data = data_filtered
    # Separate classes
    minority_class = data[data[cls] == data[cls].value_counts().idxmin()]
    majority_class = data[data[cls] == data[cls].value_counts().idxmax()]

    # Undersample majority class
    majority_class_sampled = majority_class.sample(n=len(minority_class), random_state=42)

    # Combine the balanced classes
    balanced_data = pd.concat([minority_class, majority_class_sampled])

    # Shuffle the balanced data
    balanced_data = balanced_data.sample(frac=1, random_state=42).reset_index(drop=True)
    return balanced_data

df_2 = undersampling(df, cls)

X = df_2['sequence']
y = df_2[cls].values


FRS or Chelator? FRS


In [2]:
X.shape

(458,)

In [3]:
def separate(X,y, escolha):
    data = pd.DataFrame(X)
    data[escolha] = y
    # Separate data by FRS label
    data_label_0 = data[data[escolha] == 0].reset_index(drop=True)
    data_label_1 = data[data[escolha] == 1].reset_index(drop=True)
    return data_label_0, data_label_1

df_0, df_1 = separate(X, y, cls)

In [4]:
df_0.shape

(229, 2)

In [5]:
df_1.shape

(229, 2)

In [6]:
def mixup_df(df_label_0, df_label_1, cls):
    augmented_mixup_data_0 = []
    augmented_mixup_data_1 = []

    for i, row_a in df_label_0.iterrows():
        for j, row_b in df_label_0.iterrows():
            seq1 = row_a['sequence']
            seq2 = row_b['sequence']
            y = 0
            augmented_mixup_data_0.append({
                                        'sequence1': seq1,
                                        'sequence2': seq2,
                                        cls: y,
                                        })

    for i, row_a in df_label_1.iterrows():
        for j, row_b in df_label_1.iterrows():
            seq1 = row_a['sequence']
            seq2 = row_b['sequence']
            y = 1
            augmented_mixup_data_1.append({
                                        'sequence1': seq1,
                                        'sequence2': seq2,
                                        cls: y,
                                        })

    augmented_mixup_df_same_label_0 = pd.DataFrame(augmented_mixup_data_0)
    augmented_mixup_df_same_label_1 = pd.DataFrame(augmented_mixup_data_1)
    augmented_mixup_df_same_label_0['sequence'] = augmented_mixup_df_same_label_0['sequence1'].astype(str) + augmented_mixup_df_same_label_0['sequence2'].astype(str)
    augmented_mixup_df_same_label_1['sequence'] = augmented_mixup_df_same_label_1['sequence1'].astype(str) + augmented_mixup_df_same_label_1['sequence2'].astype(str)
    augmented_mixup_df = pd.concat([augmented_mixup_df_same_label_0, augmented_mixup_df_same_label_1])
    return augmented_mixup_df

df_augmented = mixup_df(df_0, df_1, cls)

In [7]:
frs_counts = df_augmented[cls].value_counts()
print(frs_counts)

FRS
0    52441
1    52441
Name: count, dtype: int64


In [8]:
df_augmented.drop_duplicates(subset=['sequence'], inplace=True)
print(df_augmented[cls].value_counts())

FRS
1    52434
0    51982
Name: count, dtype: int64


In [9]:
df_class_0 = df_augmented[df_augmented[cls] == 0]
df_class_1 = df_augmented[df_augmented[cls] == 1]

df_sampled_0 = df_class_0.sample(n=20000, random_state=42)
df_sampled_1 = df_class_1.sample(n=20000, random_state=42)

df_sampled = pd.concat([df_sampled_0, df_sampled_1]).sample(frac=1, random_state=42).reset_index(drop=True)

print(df_sampled[cls].value_counts())

FRS
1    20000
0    20000
Name: count, dtype: int64


In [10]:
df_sampled.to_csv('01_AO_db_augmented.csv')